## Example 1: Read a project's pyproject.toml

In [ ]:
import importlib.metadata, tempfile, textwrap
from pathlib import Path
import flit
from flit.config import read_flit_config

print("flit", flit.__version__)
assert flit.__version__ == importlib.metadata.version("flit")


def make_project():
    """Write a one-module project that flit can build; the later examples reuse this."""
    folder = Path(tempfile.mkdtemp(prefix="flit_nb_"))
    (folder / "pyproject.toml").write_text(textwrap.dedent("""\
        [build-system]
        requires = ["flit_core >=3.2,<5"]
        build-backend = "flit_core.buildapi"

        [project]
        name = "tiny_demo"
        authors = [{name = "Example Author", email = "author@example.com"}]
        readme = "README.md"
        requires-python = ">=3.8"
        dependencies = ["requests >=2.0"]
        dynamic = ["version", "description"]

        [project.optional-dependencies]
        test = ["pytest"]

        [project.scripts]
        tiny-demo = "tiny_demo:main"
        """))
    (folder / "README.md").write_text("# tiny_demo\n\nA demo package.\n")
    (folder / "tiny_demo.py").write_text('"""A tiny demo module."""\n__version__ = "0.3.1"\n\n\ndef main():\n    print("hi")\n')
    return folder


project = make_project()
config = read_flit_config(project / "pyproject.toml")  # reads [project] and checks it
print("module:", config.module, "| dynamic:", sorted(config.dynamic_metadata))
print("requires_dist:", config.metadata["requires_dist"])
print("entry points:", config.entrypoints)
assert config.module == "tiny_demo" and config.metadata["name"] == "tiny_demo"
assert sorted(config.dynamic_metadata) == ["description", "version"]  # taken from the module at build time
assert config.metadata["requires_python"] == ">=3.8"
assert config.metadata["description_content_type"] == "text/markdown"  # guessed from README.md
assert config.reqs_by_extra == {".none": ["requests >=2.0"], "test": ["pytest"]}, config.reqs_by_extra
assert config.entrypoints == {"console_scripts": {"tiny-demo": "tiny_demo:main"}}
print("OK -- flit read the project's pyproject.toml: module, requirements, extras and scripts")

## Example 2: flit's metadata validators

In [ ]:
from flit import validate
from flit.config import ConfigError, read_flit_config

# Each validator returns a list of problems; an empty list means valid.
assert validate.validate_name({"name": "tiny_demo"}) == []
assert validate.validate_name({"name": "bad name!"}) == ["Invalid name: 'bad name!'"]
assert validate.validate_requires_python({"requires_python": ">=3.8, <4"}) == []
assert validate.validate_requires_python({"requires_python": "3.8+"}) == ["Invalid requires-python: '3.8+'"]
assert validate.validate_requires_dist({"requires_dist": ['numpy >=1.20; python_version >= "3.9"']}) == []
bad_requirement = validate.validate_requires_dist({"requires_dist": ['numpy; python_ver >= "3.9"']})
assert bad_requirement and "python_ver" in bad_requirement[0], bad_requirement
assert validate.validate_entrypoints({"console_scripts": {"ok": "pkg.cli:main"}}) == []
assert validate.validate_entrypoints({"console_scripts": {"bad": "pkg-cli:main"}}) == [
    "Invalid entry point in group console_scripts: bad = pkg-cli:main"]
assert validate.validate_url("ftp://example.com") == ["URL 'ftp://example.com' doesn't start with https:// or http://"]
print("OK -- name, requires-python, requirements, entry points and URLs are each checked")

# read_flit_config() runs the same checks and refuses a project with an invalid value.
project = make_project()  # from Example 1
pyproject = project / "pyproject.toml"
pyproject.write_text(pyproject.read_text().replace('tiny-demo = "tiny_demo:main"', 'tiny-demo = "tiny-demo:main"'))
try:
    read_flit_config(pyproject)
    raise AssertionError("an invalid entry point was accepted")
except ConfigError as error:
    print("refused:", error)
print("OK -- read_flit_config() refused the project with an invalid entry point")

## Example 3: Build a wheel and check its metadata and RECORD

In [ ]:
import base64, configparser, csv, hashlib, io, zipfile
from flit.wheel import make_wheel_in

project = make_project()  # from Example 1
(project / "dist").mkdir()
wheel_info = make_wheel_in(project / "pyproject.toml", project / "dist")
print("built:", wheel_info.file.name)
assert wheel_info.file.name == "tiny_demo-0.3.1-py3-none-any.whl"  # version read from the module
with zipfile.ZipFile(wheel_info.file) as wheel:
    names = sorted(wheel.namelist())
    print("files:", names)
    metadata = wheel.read("tiny_demo-0.3.1.dist-info/METADATA").decode()
    entry_points = wheel.read("tiny_demo-0.3.1.dist-info/entry_points.txt").decode()
    record = list(csv.reader(io.StringIO(wheel.read("tiny_demo-0.3.1.dist-info/RECORD").decode())))
    for path, digest, size in record:
        if digest:  # every file except RECORD itself is listed with its sha256 and size
            data = wheel.read(path)
            expected = "sha256=" + base64.urlsafe_b64encode(hashlib.sha256(data).digest()).rstrip(b"=").decode()
            assert digest == expected and int(size) == len(data), path
headers = [line for line in metadata.splitlines() if ": " in line]
for line in headers:
    print("  " + line)
assert "tiny_demo.py" in names and "tiny_demo-0.3.1.dist-info/WHEEL" in names
assert "Name: tiny_demo" in headers and "Version: 0.3.1" in headers
assert "Summary: A tiny demo module." in headers  # the module docstring
assert "Requires-Dist: requests >=2.0" in headers and 'Requires-Dist: pytest ; extra == "test"' in headers
assert "Requires-Python: >=3.8" in headers and "Provides-Extra: test" in headers
scripts = configparser.ConfigParser()
scripts.read_string(entry_points)
assert scripts["console_scripts"]["tiny-demo"] == "tiny_demo:main", entry_points
print(f"OK -- the wheel carries the module, its metadata and a RECORD whose {len(record) - 1} hashes match")

## Example 4: Build an sdist without version control

In [ ]:
import tarfile
from flit.sdist import SdistBuilder

project = make_project()  # from Example 1
(project / "notes.txt").write_text("not referenced by pyproject.toml\n")
builder = SdistBuilder.from_ini_path(project / "pyproject.toml", use_vcs=False)
sdist = builder.build(project / "dist")
print("built:", sdist.name)
assert sdist.name == "tiny_demo-0.3.1.tar.gz"
with tarfile.open(sdist) as archive:
    members = sorted(archive.getnames())
    pkg_info = archive.extractfile("tiny_demo-0.3.1/PKG-INFO").read().decode()
print("members:", members)
assert {"tiny_demo-0.3.1/pyproject.toml", "tiny_demo-0.3.1/tiny_demo.py", "tiny_demo-0.3.1/README.md",
        "tiny_demo-0.3.1/PKG-INFO"} <= set(members), members
# Without version control, an sdist holds what the build needs: notes.txt is not referenced, so it is left out.
assert "tiny_demo-0.3.1/notes.txt" not in members
assert "Version: 0.3.1" in pkg_info.splitlines()
print("OK -- the sdist holds pyproject.toml, the module, the README and PKG-INFO, and nothing unreferenced")

## Example 5: The `flit build` command line

In [ ]:
import subprocess, sys, zipfile

project = make_project()  # from Example 1
version = subprocess.run([sys.executable, "-m", "flit", "--version"], capture_output=True, text=True, check=True)
print("flit --version:", (version.stdout or version.stderr).strip())
built = subprocess.run([sys.executable, "-m", "flit", "build", "--format", "wheel"],
                       cwd=project, capture_output=True, text=True)
print(built.stderr.strip()[-400:])
assert built.returncode == 0, built.stderr
wheels = sorted(path.name for path in (project / "dist").glob("*.whl"))
assert wheels == ["tiny_demo-0.3.1-py3-none-any.whl"], wheels
with zipfile.ZipFile(project / "dist" / wheels[0]) as wheel:
    assert "tiny_demo.py" in wheel.namelist()

# A broken project fails the command with a message, not a traceback.
(project / "tiny_demo.py").write_text("def main():\n    pass\n")  # no docstring and no __version__
failed = subprocess.run([sys.executable, "-m", "flit", "build", "--format", "wheel"],
                        cwd=project, capture_output=True, text=True)
print("broken project ->", failed.returncode, failed.stderr.strip().splitlines()[-1])
assert failed.returncode != 0 and "Traceback" not in failed.stderr
print("OK -- `flit build --format wheel` builds the wheel, and refuses a module with no docstring or version")